In [1]:
from langgraph.graph import StateGraph  , START , END
from dotenv import load_dotenv
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI


In [2]:
load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=1.0, 
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [3]:
class BlogState(TypedDict):

    title: str 
    outline: str 
    content: str 

In [4]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title 
    title = state['title']

    # call LLM gen Outline
    prompt = f"Generate an outline for a Blog for the topic  -> {title}"
    outline = model.invoke(prompt).content

    # update State 
    state['outline']= outline
    return state 

In [5]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f"Generate a Detailed Blog for the title  -> {title} using teh following outline -> {outline}"

    content = model.invoke(prompt).content
    state['content'] = content 

    return state 



In [7]:
graph = StateGraph(BlogState)

# nodes 


graph.add_node('create_outline' , create_outline)
graph.add_node('create_blog' , create_blog)


# edges

graph.add_edge(START , 'create_outline')
graph.add_edge('create_outline' , 'create_blog')
graph.add_edge('create_blog' , END )

workflow = graph.compile()

In [9]:
initial_state = {'title' : 'Rise Of Love For the Delhi Based Rappers Seedhe Maut in India '}
final_state = workflow.invoke(initial_state)


In [10]:
from pprint import pprint
pprint(final_state)

{'content': "Here's a detailed blog post following your outline, capturing the "
            "essence of Seedhe Maut's incredible journey and the profound "
            'connection they share with their audience.\n'
            '\n'
            '---\n'
            '\n'
            '## From Underground Fire to Mainstream Adoration: The Unstoppable '
            'Rise of Seedhe Maut\n'
            '\n'
            '**I. Introduction: The Roar of the Mosh Pit – Setting the '
            'Scene**\n'
            '\n'
            'Picture this: A packed arena, lights flashing, the air thick with '
            'anticipation. Then, the beat drops – a menacing, bass-heavy '
            'rumble that vibrates through your chest. Two figures emerge, '
            'silhouetted against a pulsing backdrop, and the crowd erupts. '
            'Thousands of voices merge into a single, deafening roar, singing '
            'along to every word, every hook, every explosive punchline. Hands '
            